# Voit-on venir la faillite ?

**Projet du cours « Python pour la data science » (ENSAE, 2A)** – Lucas Poignant, Olivier Caffier, Claudia Aguado

**Problématique.** Les entreprises défaillantes présentent-elles des signaux financiers dégradés dans les années
qui précèdent leur défaillance, et ces signaux sont-ils assez nets pour être repérés à l'avance ?

Ce notebook ne contient que l'argumentation, les tableaux et les graphiques : tous les calculs sont faits par
les fonctions du dossier `src/`. Il relit les résultats intermédiaires enregistrés sur le stockage S3 du SSP Cloud
(lecture publique, sans clé). La chaîne complète, depuis les données brutes, est décrite dans le `README.md`.

In [ ]:
# Fonctions du projet (dossier src/) et réglages d'affichage
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import config
from src.analyse import graphiques, heterogeneite, portrait, regles_alerte, silence_comptable, temoins, trajectoires
from src.collecte import contexte
from src.nettoyage import base_analyse, nettoyage_bodacc

pd.set_option("display.max_columns", 30)

## Introduction

In [ ]:
df_serie = contexte.lire_serie_defaillances()
fig, ax = graphiques.tracer_courbe(
    df_serie, "date", "nombre_defaillances",
    "Défaillances d'entreprises en France, par trimestre",
    "Nombre de défaillances (données brutes)",
    f"INSEE, série {config.IDBANK_DEFAILLANCES} (par date de jugement)",
)
debut_covid = pd.Timestamp(f"{config.ANNEE_DEBUT_JUGEMENTS}-01-01")
fin_covid = pd.Timestamp(f"{config.ANNEE_FIN_PERIODE_COVID}-12-31")
graphiques.surligner_periode(ax, debut_covid, fin_covid, "période Covid")

> **À rédiger par Claudia.** Points à couvrir :
> - le contexte : l'évolution des défaillances, le creux de la période Covid et le rattrapage ensuite ;
> - l'enjeu : pourquoi repérer tôt une entreprise en difficulté ;
> - la problématique et le plan du rapport.

## Partie 1 – Les données

> **À rédiger par Claudia.** Points à couvrir :
> - les trois sources (BODACC, bilans INPI, SIRENE) et ce que chacune apporte ;
> - la période d'étude (jugements 2020-2024) et le traitement de la période Covid.

### 1.1 Les défaillances (BODACC) comparées à la série officielle

In [ ]:
df_jugements = portrait.lire_jugements_periode()
display(nettoyage_bodacc.comparer_avec_insee(df_jugements))

> **À rédiger par Lucas.** Points à couvrir :
> - le choix du premier jugement d'ouverture par SIREN et les expressions régulières utilisées ;
> - l'écart entre nos jugements et la série INSEE.

### 1.2 Les choix de nettoyage des bilans

> **À rédiger par Olivier.** Points à couvrir :
> - chaque choix de nettoyage et sa justification : types de bilan gardés, doublons, postes absents mis à 0, définition des ratios, winsorisation à 1 % et 99 % ;
> - les limites : pas de liquidité pour les bilans simplifiés, comptes confidentiels sans compte de résultat.

### 1.3 De tous les jugements à la base d'analyse : l'entonnoir

In [ ]:
df_defaillantes, df_bilans_defaillantes = temoins.lire_base_defaillantes()
nombre_jugements_total = base_analyse.compter_jugements_total()
display(base_analyse.construire_entonnoir(nombre_jugements_total, df_defaillantes))

### 1.4 Le groupe témoin

In [ ]:
df_reference = temoins.choisir_bilans_reference(df_defaillantes, df_bilans_defaillantes)
df_paires = silence_comptable.lire_paires()
display(temoins.construire_entonnoir_appariement(df_defaillantes, df_reference, df_paires))

df_base = trajectoires.lire_base_appariee()
display(temoins.comparer_groupes(df_base))

> **À rédiger par Claudia.** Points à couvrir :
> - l'entonnoir : à chaque étape, combien d'entreprises restent et pourquoi ;
> - l'appariement (même section NAF, même tranche de taille, même année de clôture, graine fixée) et ses limites.

## Partie 2 – Portrait des entreprises défaillantes

In [ ]:
# Lit SIRENE en ligne : compter 2 à 5 minutes
df_comptages = portrait.compter_entreprises_et_defaillances()
periode = f"{config.ANNEE_DEBUT_JUGEMENTS}-{config.ANNEE_FIN_JUGEMENTS}"
titre_axe_taux = "Défaillances pour 1 000 sociétés"
source_portrait = "BODACC, INSEE (SIRENE) ; calculs des auteurs"

### 2.1 Selon le secteur

In [ ]:
df_taux_secteurs = portrait.calculer_taux(df_comptages, "libelle_section")
df_taux_secteurs = df_taux_secteurs.sort_values("taux_pour_mille", ascending=False)
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_secteurs, "libelle_section", "taux_pour_mille",
    f"Taux annuel de défaillance par secteur, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par secteur.

### 2.2 Selon l'âge

In [ ]:
df_taux_ages = portrait.calculer_taux(df_comptages, "tranche_age")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_ages, "tranche_age", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'âge de la société, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par âge.

### 2.3 Selon la taille

In [ ]:
df_taux_tailles = portrait.calculer_taux(df_comptages, "groupe_effectifs")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_tailles, "groupe_effectifs", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'effectif salarié, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par taille.

### 2.4 Selon le département

In [ ]:
df_taux_departements = portrait.calculer_taux(df_comptages, "departement")
gdf_departements = portrait.preparer_carte(df_taux_departements)
fig, ax = graphiques.tracer_carte(
    gdf_departements, "taux_pour_mille",
    f"Taux annuel de défaillance par département, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la carte.

### 2.5 Les types de procédure

In [ ]:
fig, ax = graphiques.tracer_barres_empilees(
    portrait.repartir_procedures(df_jugements),
    "Répartition des premiers jugements par type de procédure", "Part des jugements (%)",
    "BODACC ; calculs des auteurs",
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la répartition des procédures.

## Partie 3 – Les trajectoires des ratios de N-4 à N-1

### 3.1 Défaillantes contre témoins

In [ ]:
figures_trajectoires = trajectoires.tracer_trajectoires_par_periode(
    df_base, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios"
)

In [ ]:
# Médianes de chaque ratio, par groupe et par horizon (les deux périodes ensemble)
df_quartiles = trajectoires.calculer_quartiles(df_base, config.RATIOS_TRAJECTOIRES, ["groupe"])
display(trajectoires.presenter_medianes(df_quartiles, ["groupe"]).round(3))

> **À rédiger par Olivier.** Points à couvrir :
> - quel ratio décroche en premier, et à quel horizon (critère de validation de l'étape 8) ;
> - les différences entre la période Covid et l'après-Covid.

### 3.2 Vérification : les entreprises présentes aux quatre horizons

In [ ]:
df_base_complete = trajectoires.garder_entreprises_completes(df_base)
figures_completes = trajectoires.tracer_trajectoires_par_periode(
    df_base_complete, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios, entreprises présentes de N-4 à N-1"
)

> **À rédiger par Olivier.** Points à couvrir :
> - ce que la vérification confirme ou nuance dans la lecture du 3.1.

### 3.3 Les rangs sectoriels à N-1 et N-2

In [ ]:
df_base_rangs = trajectoires.ajouter_rangs(df_base, trajectoires.lire_rangs_sectoriels())
display(trajectoires.resumer_rangs(df_base_rangs))

> **À rédiger par Olivier.** Points à couvrir :
> - la position des défaillantes dans leur secteur, et pourquoi les témoins sont proches de 50.

### 3.4 Les situations extrêmes

In [ ]:
display(trajectoires.compter_situations_extremes(df_base, ["groupe"]))

> **À rédiger par Olivier.** Points à couvrir :
> - capitaux propres négatifs et trésorerie quasi nulle : à quel horizon l'écart se creuse.

## Partie 4 – Des trajectoires différentes selon les entreprises

In [ ]:
df_sous_groupes = heterogeneite.ajouter_sous_groupes(df_base, heterogeneite.construire_sous_groupes())

### 4.1 Selon le grand secteur

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "grand_secteur")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.2 Selon l'âge au jugement

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "classe_age")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.3 Selon la taille

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "taille_appariement")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.4 Selon le type de procédure

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "type_procedure")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

## Partie 5 – Le silence comptable

In [ ]:
df_toutes_defaillantes = silence_comptable.lire_defaillantes()
display(silence_comptable.decrire_population_totale(df_toutes_defaillantes, df_paires))

In [ ]:
df_grille = silence_comptable.construire_grille(df_paires, df_toutes_defaillantes)
df_grille = silence_comptable.ajouter_statut(df_grille, df_base)
display(silence_comptable.calculer_taux_silence(df_grille, ["groupe"]))
figures_silence = silence_comptable.tracer_silence_par_periode(df_grille)

In [ ]:
# Biais de sélection : ratios à N-2 des défaillantes, selon qu'elles publient encore à N-1
display(silence_comptable.comparer_publiantes_et_silencieuses(df_base))

> **À rédiger par Olivier.** Points à couvrir :
> - l'ampleur du silence comptable avant le jugement, défaillantes contre témoins ;
> - le biais de sélection que ce silence crée pour la partie 3 (critère de validation de l'étape 10).

## Partie 6 – Des règles d'alerte simples

In [ ]:
df_alertes = regles_alerte.ajouter_alertes(df_base.copy())
display(regles_alerte.calculer_couverture(df_alertes))
display(regles_alerte.tableau_performances(df_alertes))
display(regles_alerte.tableau_performances_par_periode(df_alertes))

> **À rédiger par Lucas.** Points à couvrir :
> - les taux de détection et de fausses alertes à N-1 et N-2 ;
> - pourquoi les fausses alertes seraient plus nombreuses en population réelle (critère de validation de l'étape 11).

## Conclusion et limites

> **À rédiger par Claudia.** Points à couvrir :
> - la réponse à la problématique ;
> - les limites : silence comptable et biais de sélection, appariement, valeurs actuelles des effectifs dans SIRENE, période Covid, comptes récents pas encore déposés.